# 01 — Exploratory Data Analysis (training split only)

**Rule for this notebook:** it reads the *training* split only, via `load_train`.
The held-out test set is never loaded here. Looking at it now — even just its
default rate or a histogram — would let what we see leak into feature and
threshold decisions, and the final test number would stop being an honest estimate.

Every section states a hypothesis *before* the code that checks it, and ends
with what the answer means for a later module (`features.py`, `evaluate.py`,
`fairness.py`). Fill in the **Finding** lines after running.

Prerequisite: `data/raw/application_train.csv` exists and `python -m src.data`
has been run (creates `data/processed/split.csv`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Run from the repo root so `src` imports and config paths resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.data import load_config, load_train

cfg = load_config()
d = cfg["data"]
TARGET, ID, PROTECTED = d["target_column"], d["id_column"], d["protected_attribute"]
EXPOSURE = cfg["cost_assumptions"]["exposure_column"]

X, y = load_train(cfg)
df = X.assign(**{TARGET: y})
print(f"training rows: {len(df):,}   columns: {X.shape[1]}")

## 1. Base rate

**Hypothesis:** about 8% of training applicants defaulted. If so, a model that
approves everyone scores ~92% accuracy while catching zero defaults — the
reason accuracy is reported once (baseline row) and then abandoned for PR-AUC.

In [ ]:
base_rate = y.mean()
print(f"default rate: {base_rate:.2%}")
print(f"accuracy of 'predict no default for everyone': {1 - base_rate:.2%}")

**Finding:** _fill in_

**Carries into:** `evaluate.py` — PR-AUC of a no-skill model equals this base
rate, so it is the floor every real model must beat.

## 2. Column types and categorical cardinality

**Hypothesis:** most columns are numeric; roughly 16 are text (`object`)
categoricals, almost all with under 20 levels, with `ORGANIZATION_TYPE` the
exception (~58 levels). Low cardinality means one-hot encoding in Branch A
stays a manageable width.

Watch for numeric columns that are really categories (the `FLAG_*` columns are
0/1 integers — fine to leave numeric).

In [ ]:
print(X.dtypes.value_counts(), "\n")

cat_cols = X.select_dtypes(include="object").columns.tolist()
cardinality = X[cat_cols].nunique(dropna=False).sort_values(ascending=False)
print(f"{len(cat_cols)} categorical columns (levels incl. NaN):")
print(cardinality.to_string())

**Finding:** _fill in_

**Carries into:** `features.py` — this list is the starting point for the single
categorical-column list both branches share. Copy it in explicitly rather than
re-detecting by dtype at fit time, so a schema surprise fails loudly.

## 3. Missingness — and whether it carries signal

**Hypothesis:** the housing-description columns (`*_AVG`, `*_MODE`, `*_MEDI`)
are 50–70% missing, and `EXT_SOURCE_1` / `OWN_CAR_AGE` are also mostly missing.
We further expect *missing* itself to predict default (applicants with less
paperwork on file default more). If that holds, it is evidence for the
LightGBM branch passing nulls through instead of imputing — imputing a median
erases exactly that signal.

In [ ]:
missing = X.isna().mean().sort_values(ascending=False)
print(f"columns with any missing: {(missing > 0).sum()} of {X.shape[1]}")
print(f"columns >50% missing:     {(missing > 0.5).sum()}\n")
print(missing.head(15).map("{:.1%}".format).to_string())

In [ ]:
# Default rate when a column is missing vs present, for the most-missing
# columns plus a few we care about by name.
check = list(missing[missing > 0.3].index[:8]) + ["EXT_SOURCE_1", "EXT_SOURCE_3", "OWN_CAR_AGE"]
check = list(dict.fromkeys(c for c in check if c in X.columns))  # dedupe, keep order

rows = []
for col in check:
    is_na = X[col].isna()
    rows.append({
        "column": col,
        "missing_share": is_na.mean(),
        "default_if_missing": y[is_na].mean(),
        "default_if_present": y[~is_na].mean(),
    })
miss_signal = pd.DataFrame(rows).set_index("column")
miss_signal.style.format("{:.2%}")

**Finding:** _fill in — is the default rate noticeably different when missing?_

**Carries into:** `features.py` — Branch B passes nulls through. For Branch A
(median imputation), consider whether a missing-indicator is worth adding for
the columns where the gap is large; that would be one engineered feature with
this table as its stated hypothesis.

## 4. The `DAYS_EMPLOYED = 365243` placeholder

`DAYS_*` columns count days *before* the application, so they are negative.
`365243` (≈1000 years in the future) is not a real value — it is a code.

**Hypothesis:** these rows are almost entirely pensioners/unemployed
(`NAME_INCOME_TYPE`), make up ~18% of the data, and default *less* than
everyone else. If so, the placeholder holds real information ("not employed,
retired") that must be kept as a flag — while the raw number itself must be
removed, or every scaled/linear model treats it as an extreme employment length.

In [ ]:
PLACEHOLDER = 365243
is_ph = X["DAYS_EMPLOYED"] == PLACEHOLDER
print(f"placeholder rows: {is_ph.sum():,} ({is_ph.mean():.1%})")
print(f"default rate — placeholder: {y[is_ph].mean():.2%}   others: {y[~is_ph].mean():.2%}\n")
print("NAME_INCOME_TYPE among placeholder rows:")
print(X.loc[is_ph, "NAME_INCOME_TYPE"].value_counts().to_string())
print(f"\nany other positive DAYS_EMPLOYED values? {((X['DAYS_EMPLOYED'] > 0) & ~is_ph).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
X["DAYS_EMPLOYED"].plot.hist(bins=60, ax=axes[0], title="DAYS_EMPLOYED as stored")
(X.loc[~is_ph, "DAYS_EMPLOYED"] / -365).plot.hist(
    bins=60, ax=axes[1], title="Years employed, placeholder removed")
axes[1].set_xlabel("years")
plt.tight_layout()

**Finding:** _fill in_

**Carries into:** `features.py` — engineered feature #1: an
`EMPLOYED_PLACEHOLDER` flag plus `DAYS_EMPLOYED` set to NaN for those rows.
This is a fixed rule (no statistic learned from data), so doing it inside the
Pipeline is leak-free; it still belongs in the Pipeline so `app.py` applies it too.

## 5. The protected attribute: `CODE_GENDER`

CLAUDE.md says to verify the actual values rather than assume.

**Hypothesis:** values are `F`, `M`, and a handful of `XNA`; women are the
majority of applicants; men have a *higher* default rate. If base rates differ,
that is precisely the condition under which demographic parity and equalized
odds cannot both hold — so this table is the evidence for the tradeoff
argument in `fairness.py`.

In [ ]:
gender = (
    df.groupby(PROTECTED, dropna=False)[TARGET]
    .agg(applicants="size", default_rate="mean")
    .assign(share=lambda t: t["applicants"] / t["applicants"].sum())
)
gender.style.format({"default_rate": "{:.2%}", "share": "{:.1%}", "applicants": "{:,}"})

**Finding:** _fill in — exact values, counts, and the default-rate gap._

**Team decisions this forces (record them in `config.yaml` / `fairness.py`):**
- What happens to tiny groups (e.g. `XNA`)? With only a few rows, any group
  metric for them is noise. Options: report them but exclude from the gap, or
  drop the rows with a stated reason. Don't merge them into `F`/`M` silently.
- `fairness.protected_in_model`: is `CODE_GENDER` a model input at all?
- `fairness.primary_metric`: given the base-rate gap above, which one, and what
  does the team accept giving up?

## 6. Loan size (`AMT_CREDIT`) — the profit exposure

**Hypothesis:** `AMT_CREDIT` is strongly right-skewed (a few very large loans),
and its relationship to default is weak or slightly negative. Since profit and
loss are both rates × `AMT_CREDIT`, a handful of large loans will dominate the
profit curve — so the team should look at the profit sweep with that in mind,
and a small-loan default costs far less than a large-loan default.

In [ ]:
print(X[EXPOSURE].describe().map("{:,.0f}".format).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
X[EXPOSURE].plot.hist(bins=60, ax=axes[0], title=f"{EXPOSURE}")
deciles = pd.qcut(X[EXPOSURE], 10, duplicates="drop")
y.groupby(deciles, observed=True).mean().plot.bar(
    ax=axes[1], title=f"Default rate by {EXPOSURE} decile")
axes[1].axhline(base_rate, ls="--", c="k", lw=1)
axes[1].set_xticklabels([]); axes[1].set_xlabel("decile (small → large)")
plt.tight_layout()

top10_share = X[EXPOSURE].nlargest(len(X) // 10).sum() / X[EXPOSURE].sum()
print(f"\nlargest 10% of loans hold {top10_share:.1%} of total credit exposure")

**Finding:** _fill in_

**Carries into:** `config.yaml` → `cost_assumptions`. When the team picks
`loss_given_default` and `expected_interest`, use these numbers to sanity-check
the totals (e.g. "at 8% defaults and LGD 0.6, approving everyone earns/loses
roughly X").

## 7. Strongest single signals: `EXT_SOURCE_1/2/3`

These are external credit-bureau-style scores, scaled 0–1.

**Hypothesis:** each one alone sorts applicants by risk better than any other
raw column — default rate should fall steadily from the lowest to the highest
decile. If that holds, any model that doesn't beat a single EXT_SOURCE column
by a clear PR-AUC margin is not earning its complexity.

In [ ]:
ext = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in X.columns]
fig, axes = plt.subplots(1, len(ext), figsize=(4 * len(ext), 3.2), sharey=True)
for ax, col in zip(np.atleast_1d(axes), ext):
    bins = pd.qcut(X[col], 10, duplicates="drop")
    y.groupby(bins, observed=True).mean().plot.bar(ax=ax, title=col)
    ax.axhline(base_rate, ls="--", c="k", lw=1)
    ax.set_xticklabels([]); ax.set_xlabel("decile (low → high)")
axes_flat = np.atleast_1d(axes)
axes_flat[0].set_ylabel("default rate")
plt.tight_layout()

# Pairwise overlap: if they are highly correlated, they're one signal, not three.
print(X[ext].corr().round(2))

**Finding:** _fill in_

**Carries into:** `features.py` — candidate engineered feature: a combination of
the EXT_SOURCE scores (e.g. their mean, ignoring missing ones). State the
hypothesis ("the combined score separates risk better than any single one")
before adding it, and confirm it with CV, not with this notebook.

## 8. Summary — decisions handed to the next modules

| # | Question | Finding | Goes to |
|---|---|---|---|
| 1 | Base rate | _ | `evaluate.py` (PR-AUC floor) |
| 2 | Categorical columns + cardinality | _ | `features.py` column lists |
| 3 | Does missingness predict default? | _ | `features.py` (null passthrough / indicators) |
| 4 | `DAYS_EMPLOYED` placeholder | _ | `features.py` engineered feature #1 |
| 5 | `CODE_GENDER` values + base-rate gap | _ | `fairness.py`, `config.yaml` |
| 6 | Loan-size skew | _ | `config.yaml` cost assumptions |
| 7 | EXT_SOURCE strength | _ | `features.py` candidate feature |

Nothing in this notebook was fit, and the test split was never loaded.